# COMSATS University Islamabad
## Lahore Campus

# Fitness Calorie Burn Prediction System
### Machine Learning Fundamentals

---

## Student Information

| Detail | Information |
|---|---|
| **Student Name** | Aima Mushtaq |
| **Registration No.** | SP24-BSE-009 |
| **Section** | BSE-B |
| **Course Name** | Machine Learning Fundamentals |
| **Submitted To** | Dr. Rao Adeel |

---

## Live Application Link

### Access the Fitness Calorie Burn Prediction System

[Click Here to Open the Live Application](PASTE_YOUR_STREAMLIT_APP_LINK_HERE)

---

### Project Description

This project implements a **Fitness Calorie Burn Prediction System** using Machine Learning regression algorithms. The system uses fitness and exercise-related attributes to predict the numerical amount of calories burned by a person.

Multiple regression algorithms are trained and compared to determine the best-performing model. The selected model is then used in a Streamlit web application where users can enter their fitness information and obtain a predicted calorie burn value.

---

# Fitness Calorie Burn Prediction — Regression

**Purpose:** Train and compare regression models that predict calories burned during exercise.

This notebook:
1. Loads `calories.csv`.
2. Checks and prepares the data.
3. Splits features (`X`) and target (`y`).
4. Trains and compares regression algorithms.
5. Saves the best model as `best_fitness_model.pkl`.
6. Saves comparison results as `fitness_model_comparison.csv`.

**Before running:** Put `calories.csv` in the same folder as this notebook. Run the cells from top to bottom.

In [ ]:
# Step 1: Import the libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print("Libraries imported successfully.")

In [ ]:
# Step 2: Load the dataset

data = pd.read_csv("calories.csv")

print("Dataset shape:", data.shape)
display(data.head())
print("\nColumn names:")
print(data.columns.tolist())

In [ ]:
# Step 3: Clean column names and identify the target column

# Remove extra spaces from column names.
data.columns = data.columns.str.strip()

# Find the target column even if its capitalization differs.
target_matches = [column for column in data.columns
                  if column.lower() in ["calories", "calories_burned"]]

if len(target_matches) == 0:
    raise ValueError(
        "Could not find a target column named Calories or Calories_Burned. "
        "Please check the column names printed in Step 2."
    )

target_column = target_matches[0]

# Remove rows where the target value is missing.
data = data.dropna(subset=[target_column]).copy()

# Remove an ID column if present, because it is an identifier rather than an exercise measurement.
id_columns = [column for column in data.columns
              if column.lower() in ["user_id", "user id", "id"]]
data = data.drop(columns=id_columns, errors="ignore")

print("Target column:", target_column)
print("Rows and columns after basic cleaning:", data.shape)
display(data.head())

In [ ]:
# Step 4: Separate features (X) and target (y)

X = data.drop(columns=[target_column])
y = data[target_column]

# Identify text/category columns and numeric columns automatically.
categorical_columns = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()

# Remove any columns that contain no usable values.
numeric_columns = [column for column in numeric_columns if X[column].notna().any()]
categorical_columns = [column for column in categorical_columns if X[column].notna().any()]

# Keep only columns that can be processed.
selected_columns = numeric_columns + categorical_columns
X = X[selected_columns]

print("Feature columns:", X.columns.tolist())
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)
print("Target:", target_column)

In [ ]:
# Step 5: Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

In [ ]:
# Step 6: Prepare missing values and categorical data

numeric_preparation = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_preparation = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preparation_steps = []

if numeric_columns:
    preparation_steps.append(("numeric", numeric_preparation, numeric_columns))

if categorical_columns:
    preparation_steps.append(("categorical", categorical_preparation, categorical_columns))

preprocessor = ColumnTransformer(transformers=preparation_steps)

print("Preprocessing setup is ready.")

In [ ]:
# Step 7: Define regression algorithms

models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42, max_depth=12),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

print("Models ready:", list(models.keys()))

In [ ]:
# Step 8: Compare models using 5-fold cross-validation

results = []

for model_name, model in models.items():
    model_pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    # Negative mean absolute error is returned by scikit-learn,
    # so multiply by -1 to make the result positive.
    cv_scores = cross_val_score(
        model_pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    results.append({
        "Model": model_name,
        "Average CV MAE": -cv_scores.mean()
    })

comparison = pd.DataFrame(results)
comparison = comparison.sort_values(by="Average CV MAE").reset_index(drop=True)

display(comparison)
comparison.to_csv("fitness_model_comparison.csv", index=False)

print("Saved: fitness_model_comparison.csv")

In [ ]:
# Step 9: Train the best model and evaluate it on the test set

best_model_name = comparison.loc[0, "Model"]
best_algorithm = models[best_model_name]

best_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", best_algorithm)
])

best_model.fit(X_train, y_train)
predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("Best model:", best_model_name)
print("Test MAE:", round(mae, 2))
print("Test RMSE:", round(rmse, 2))
print("Test R-squared:", round(r2, 3))

In [ ]:
# Step 10: Save the model in a compressed file

# Compression can reduce the file size. The app must load this file with joblib.load().
joblib.dump(best_model, "best_fitness_model.pkl", compress=3)

print("Saved: best_fitness_model.pkl")

from pathlib import Path
model_size_mb = Path("best_fitness_model.pkl").stat().st_size / (1024 * 1024)
print("Model file size:", round(model_size_mb, 2), "MB")

if model_size_mb >= 25:
    print("The model is still too large for GitHub's browser upload. "
          "Consider using a smaller model or Git LFS.")
else:
    print("The model is under 25 MB and should fit GitHub's browser upload limit.")

## Step 11: Example prediction

The next cell uses the first row of the test set as an example. It does not invent user input values; it demonstrates how the trained model makes a prediction from an existing row.

In [ ]:
example_row = X_test.iloc[[0]]
actual_calories = y_test.iloc[0]
predicted_calories = best_model.predict(example_row)[0]

display(example_row)
print("Actual calories:", round(float(actual_calories), 2))
print("Predicted calories:", round(float(predicted_calories), 2))